# Gradient Boosting from Scratch

[[adaboost]] reweights the training rows. [[xgboost]], [[lightgbm]] and
[[catboost]] all describe themselves as gradient boosting. This page is the step
in between — the idea those three libraries are optimised implementations *of*.

The idea in one sentence: **fit each new tree to the negative gradient of the
loss with respect to the current predictions.**

That sounds abstract until you compute the gradient for squared error, at which
point it becomes something much more familiar:

$$L = \tfrac{1}{2}(y - F)^2 \qquad \frac{\partial L}{\partial F} = -(y - F) \qquad -\frac{\partial L}{\partial F} = y - F = \text{the residual}$$

So "fit the next tree to the residuals" is not a separate algorithm — it is what
gradient boosting *becomes* when the loss is squared error. Change the loss and
the same machinery gives you something else, which is the whole point of the
generalisation.

Every number below is computed by hand on 12 EEG recordings and checked against
scikit-learn.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
np.set_printoptions(precision=4, suppress=True, linewidth=140)


def banner(title, ch="="):
    print(ch * 78)
    print(f"  {title}")
    print(ch * 78)


def step(label):
    print()
    print(label)
    print("=" * 78)


def check(manual, from_array, label="MATCH"):
    ok = np.allclose(manual, from_array)
    print(f"      = {manual:+.6f}   |  from array: {from_array:+.6f}   |  {label}: {ok}")
    return ok


def expand(u, v, per_line=4):
    """Write a dot product out term by term."""
    terms = [f"({a:+.4f})*({b:+.4f})" for a, b in zip(u, v)]
    lines, out = [], ""
    for i, t in enumerate(terms):
        out += t + (" + " if i < len(terms) - 1 else "")
        if (i + 1) % per_line == 0:
            lines.append(out); out = ""
    if out:
        lines.append(out)
    return "\n      ".join(lines)


def show_matrix(M, name="M", rows=None, cols=None, fmt="{:+8.3f}", max_cols=14):
    M = np.asarray(M)
    if M.ndim == 1:
        M = M[None, :]
    cut = M.shape[1] > max_cols
    Ms = M[:, :max_cols] if cut else M
    print(f"{name} {np.asarray(M).shape}" +
          (f"   [first {max_cols} of {M.shape[1]} columns]" if cut else "") + " =")
    if cols is not None:
        head = "".join(f"{str(c):>10}" for c in (cols[:max_cols] if cut else cols))
        print(" " * 13 + head)
    for r in range(Ms.shape[0]):
        lab = f"{str(rows[r]):>10} | " if rows is not None else f"{r:>10} | "
        print("   " + lab + "  ".join(fmt.format(v) for v in Ms[r]))


def show_matmul(A, B, C, na="A", nb="B", nc="C", i=0, j=0,
                rows_A=None, note=None, max_cols=14):
    """Print both operands of A @ B = C, then verify one element by hand."""
    A, B, C = np.asarray(A), np.asarray(B), np.asarray(C)
    banner(f"{na} {A.shape}   @   {nb} {B.shape}   =   {nc} {C.shape}")
    print(f"  shape rule: ({A.shape[0]} x {A.shape[1]}) @ ({B.shape[0]} x {B.shape[1]})"
          f" -> ({A.shape[0]} x {B.shape[1]})   inner dim {A.shape[1]} cancels")
    if note:
        print(f"  {note}")
    print("=" * 78)
    print()
    show_matrix(A, na, rows_A, max_cols=max_cols)
    print()
    show_matrix(B, nb, max_cols=max_cols)
    print()
    show_matrix(C, nc, rows_A, max_cols=max_cols)
    print(f"\n  worked element  {nc}[{i},{j}]  =  row {i} of {na}  .  column {j} of {nb}")
    print(f"      row {i} of {na} = {np.round(A[i], 4)}")
    print(f"      col {j} of {nb} = {np.round(B[:, j], 4)}")
    print(f"      {expand(A[i], B[:, j])}")
    check(float(A[i] @ B[:, j]), float(C[i, j]))


def show_elementwise(A, B, C, na, nb, nc, op="+", rows=None):
    A, B, C = np.asarray(A), np.asarray(B), np.asarray(C)
    banner(f"{nc} = {na} {op} {nb}      all three are {A.shape} - element-wise")
    for r in range(A.shape[0]):
        lab = f"{rows[r]:>10}" if rows is not None else f"{r:>10}"
        print(f"\n{lab} |")
        print("   " + f"{na:>14}: " + "  ".join(f"{v:+8.3f}" for v in A[r]))
        print("   " + f"{nb:>14}: " + "  ".join(f"{v:+8.3f}" for v in B[r]))
        print("   " + f"{nc:>14}: " + "  ".join(f"{v:+8.3f}" for v in C[r]))
    print(f"\n  check [0,0]: {A[0,0]:+.6f} {op} {B[0,0]:+.6f} = {C[0,0]:+.6f}   "
          f"MATCH: {np.isclose(A[0,0] + B[0,0] if op == '+' else A[0,0] - B[0,0], C[0,0])}")


print("Display helpers ready: banner, step, check, expand, show_matrix, "
      "show_matmul, show_elementwise.")

# ----------------------------------------------------------------------------
#  A deliberately tiny EEG-flavoured table: 12 recordings, hand-checkable.
# ----------------------------------------------------------------------------
feature_names = ["line_length", "spike_rate", "age"]
region_names  = ["frontal", "temporal", "occipital"]

X = np.array([
    [12.0,  0.5, 24.0],
    [15.0,  1.2, 31.0],
    [42.0,  6.8, 58.0],
    [38.0,  5.1, 47.0],
    [11.0,  0.3, 19.0],
    [55.0,  8.2, 63.0],
    [40.0,  1.9, 35.0],
    [47.0,  7.4, 52.0],
    [14.0,  0.8, 28.0],
    [51.0,  6.1, 60.0],
    [45.0,  2.4, 41.0],
    [35.0,  4.7, 30.0],
])

# categorical column, used by the CatBoost notebook
region = np.array([0, 1, 1, 2, 0, 1, 2, 1, 0, 2, 0, 1])

# binary target: did a seizure follow?
y_class = np.array([0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1])

# continuous target: seizure burden in minutes/hour
y_reg = np.array([0.2, 0.5, 6.4, 4.8, 0.1, 9.1, 3.5, 7.2, 0.3, 6.9, 1.6, 4.1])

n_samples, n_features = X.shape
rec_ids = [f"rec{i:02d}" for i in range(n_samples)]

print(f"{n_samples} recordings x {n_features} features\n")
hdr = f"{'id':>6}  {'region':>10}  " + "  ".join(f"{f:>12}" for f in feature_names)
hdr += f"  {'y_class':>8}  {'y_reg':>7}"
print(hdr)
print("-" * len(hdr))
for i in range(n_samples):
    print(f"{rec_ids[i]:>6}  {region_names[region[i]]:>10}  " +
          "  ".join(f"{v:>12.1f}" for v in X[i]) +
          f"  {y_class[i]:>8}  {y_reg[i]:>7.1f}")
print("-" * len(hdr))
print(f"\nclass balance: {int((y_class==0).sum())} negative, {int((y_class==1).sum())} positive")
print(f"y_reg mean = {y_reg.mean():.4f}, std = {y_reg.std():.4f}")

Display helpers ready: banner, step, check, expand, show_matrix, show_matmul, show_elementwise.
12 recordings x 3 features

    id      region   line_length    spike_rate           age   y_class    y_reg
-------------------------------------------------------------------------------
 rec00     frontal          12.0           0.5          24.0         0      0.2
 rec01    temporal          15.0           1.2          31.0         0      0.5
 rec02    temporal          42.0           6.8          58.0         1      6.4
 rec03   occipital          38.0           5.1          47.0         1      4.8
 rec04     frontal          11.0           0.3          19.0         0      0.1
 rec05    temporal          55.0           8.2          63.0         1      9.1
 rec06   occipital          40.0           1.9          35.0         1      3.5
 rec07    temporal          47.0           7.4          52.0         1      7.2
 rec08     frontal          14.0           0.8          28.0         0      

## 1. Three rounds, by hand

The algorithm (Friedman, 2001):

1. Start with a constant prediction $F_0$ — the value minimising the loss with no features. For squared error that is the mean of $y$.
2. For $m = 1 \dots M$:
   - compute the **pseudo-residuals** $r_i = -\partial L/\partial F(x_i)$ evaluated at the current $F_{m-1}$
   - fit a shallow regression tree to those residuals
   - update $F_m = F_{m-1} + \eta\, h_m$, where $\eta$ is the **learning rate**

The trees are deliberately weak — depth 1 or 2 — and $\eta$ is small. Each round
corrects a little of what remains.

In [2]:
from sklearn.tree import DecisionTreeRegressor, export_text

banner("GRADIENT BOOSTING, THREE ROUNDS ON 12 RECORDINGS")

eta = 0.5              # deliberately large so three rounds are visible
y = y_reg.astype(float)

step("Round 0: the constant that minimises squared error")
F = np.full(n_samples, y.mean())
print(f"  F_0 = mean(y) = ({' + '.join(f'{v:.1f}' for v in y[:4])} + ...)"
      f" / {n_samples} = {y.mean():.4f}")
print(f"  SSE = {((y - F)**2).sum():.4f}")

trees = []
for m in range(1, 4):
    step(f"Round {m}")
    r = y - F                                    # negative gradient, squared error
    print(f"  pseudo-residuals r = y - F  (this IS -dL/dF for squared error)")
    print(f"    {'id':>6}  {'y':>7}  {'F':>8}  {'r = y-F':>9}")
    for i in range(n_samples):
        print(f"    {rec_ids[i]:>6}  {y[i]:>7.2f}  {F[i]:>8.4f}  {r[i]:>9.4f}")

    t = DecisionTreeRegressor(max_depth=1, random_state=0).fit(X, r)
    trees.append(t)
    thr  = t.tree_.threshold[0]
    feat = feature_names[t.tree_.feature[0]]
    left, right = t.tree_.value[1].ravel()[0], t.tree_.value[2].ravel()[0]
    print(f"\n  tree {m}: split on {feat} <= {thr:.3f}")
    print(f"    left  leaf (<= {thr:.3f}) : {left:+.4f}")
    print(f"    right leaf (>  {thr:.3f}) : {right:+.4f}")

    # verify the leaf values are the means of the residuals in each leaf
    col = t.tree_.feature[0]
    mask = X[:, col] <= thr
    print(f"    check: mean(r | left)  = {r[mask].mean():+.4f}"
          f"   MATCH: {np.allclose(r[mask].mean(), left)}")
    print(f"           mean(r | right) = {r[~mask].mean():+.4f}"
          f"   MATCH: {np.allclose(r[~mask].mean(), right)}")

    h = t.predict(X)
    F = F + eta * h
    print(f"\n  update F <- F + {eta} * h")
    print(f"    SSE now = {((y - F)**2).sum():.4f}")

step("Three rounds of progress")
Fc = np.full(n_samples, y.mean())
print(f"  {'round':>6}  {'SSE':>10}  {'reduction':>10}")
print("  " + "-" * 30)
prev = ((y - Fc)**2).sum()
print(f"  {0:>6}  {prev:>10.4f}  {'':>10}")
for m, t in enumerate(trees, 1):
    Fc = Fc + eta * t.predict(X)
    cur = ((y - Fc)**2).sum()
    print(f"  {m:>6}  {cur:>10.4f}  {prev-cur:>10.4f}")
    prev = cur
print("  " + "-" * 30)

  GRADIENT BOOSTING, THREE ROUNDS ON 12 RECORDINGS

Round 0: the constant that minimises squared error
  F_0 = mean(y) = (0.2 + 0.5 + 6.4 + 4.8 + ...) / 12 = 3.7250
  SSE = 111.7625

Round 1
  pseudo-residuals r = y - F  (this IS -dL/dF for squared error)
        id        y         F    r = y-F
     rec00     0.20    3.7250    -3.5250
     rec01     0.50    3.7250    -3.2250
     rec02     6.40    3.7250     2.6750
     rec03     4.80    3.7250     1.0750
     rec04     0.10    3.7250    -3.6250
     rec05     9.10    3.7250     5.3750
     rec06     3.50    3.7250    -0.2250
     rec07     7.20    3.7250     3.4750
     rec08     0.30    3.7250    -3.4250
     rec09     6.90    3.7250     3.1750
     rec10     1.60    3.7250    -2.1250
     rec11     4.10    3.7250     0.3750

  tree 1: split on spike_rate <= 3.550
    left  leaf (<= 3.550) : -2.6917
    right leaf (>  3.550) : +2.6917
    check: mean(r | left)  = -2.6917   MATCH: True
           mean(r | right) = +2.6917   MATCH: Tr

## 2. Checking it against scikit-learn

The hand-built model above should be *exactly* what `GradientBoostingRegressor`
produces with the same settings. Worth confirming, because it means the mental
model is the implementation, not an analogy for it.

In [3]:
from sklearn.ensemble import GradientBoostingRegressor

banner("HAND-BUILT vs SCIKIT-LEARN")

gb = GradientBoostingRegressor(n_estimators=3, learning_rate=eta, max_depth=1,
                               loss='squared_error', random_state=0).fit(X, y)

mine = np.full(n_samples, y.mean())
for t in trees:
    mine = mine + eta * t.predict(X)
theirs = gb.predict(X)

print(f"  {'id':>6}  {'y':>7}  {'by hand':>9}  {'sklearn':>9}  {'diff':>10}")
print("  " + "-" * 48)
for i in range(n_samples):
    print(f"  {rec_ids[i]:>6}  {y[i]:>7.2f}  {mine[i]:>9.4f}"
          f"  {theirs[i]:>9.4f}  {mine[i]-theirs[i]:>10.2e}")
print("  " + "-" * 48)
print(f"  max|diff| = {np.abs(mine-theirs).max():.2e}   MATCH: "
      f"{np.allclose(mine, theirs)}")
print()
print(f"  init prediction: mine {y.mean():.6f}"
      f"  |  sklearn {gb.init_.constant_.ravel()[0]:.6f}"
      f"  |  MATCH: {np.allclose(y.mean(), gb.init_.constant_.ravel()[0])}")

  HAND-BUILT vs SCIKIT-LEARN
      id        y    by hand    sklearn        diff
  ------------------------------------------------
   rec00     0.20     1.1493     1.1493    0.00e+00
   rec01     0.50     1.1493     1.1493    0.00e+00
   rec02     6.40     6.6980     6.6980    0.00e+00
   rec03     4.80     4.9511     4.9511    0.00e+00
   rec04     0.10     1.1493     1.1493    0.00e+00
   rec05     9.10     6.6980     6.6980    0.00e+00
   rec06     3.50     2.2594     2.2594   -4.44e-16
   rec07     7.20     6.6980     6.6980    0.00e+00
   rec08     0.30     1.1493     1.1493    0.00e+00
   rec09     6.90     6.6980     6.6980    0.00e+00
   rec10     1.60     2.2594     2.2594   -4.44e-16
   rec11     4.10     3.8409     3.8409    0.00e+00
  ------------------------------------------------
  max|diff| = 4.44e-16   MATCH: True

  init prediction: mine 3.725000  |  sklearn 3.725000  |  MATCH: True


## 3. The learning rate: why shrink at all?

Nothing forces $\eta < 1$. With $\eta = 1$ each tree is applied at full strength
and the training error falls fastest. So why shrink?

Because each tree was fitted to residuals computed from the *current* model, and
it is only a good correction near that point. Taking the full step overshoots
into territory the tree knows nothing about. Small steps, many of them, track the
true function more carefully — the same argument as a learning rate in
[[optimisers-sgd-to-adam]], and it produces the same trade: lower $\eta$ needs
more rounds.

The table below shows both halves — training error and held-out error — because
they say different things.

In [4]:
from sklearn.model_selection import cross_val_score, KFold
from sklearn.datasets import make_friedman1
from sklearn.model_selection import train_test_split

banner("LEARNING RATE AND NUMBER OF ROUNDS")

Xb, yb = make_friedman1(n_samples=400, noise=1.5, random_state=0)
cv = KFold(5, shuffle=True, random_state=0)

step("Training error always prefers the bigger step")
print(f"  {'eta':>6}  " + "".join(f"{f'{m} trees':>11}" for m in [5, 20, 100]))
print("  " + "-" * 40)
for e in [0.05, 0.1, 0.5, 1.0]:
    row = []
    for M in [5, 20, 100]:
        g = GradientBoostingRegressor(n_estimators=M, learning_rate=e,
                                      max_depth=3, random_state=0).fit(Xb, yb)
        row.append(np.mean((yb - g.predict(Xb))**2))
    print(f"  {e:>6}  " + "".join(f"{v:>11.4f}" for v in row))
print("  " + "-" * 40)

step("Held-out error does not")
print(f"  {'eta':>6}  " + "".join(f"{f'{m} trees':>11}" for m in [5, 20, 100, 400]))
print("  " + "-" * 51)
best = (None, np.inf)
for e in [0.05, 0.1, 0.5, 1.0]:
    row = []
    for M in [5, 20, 100, 400]:
        s = -cross_val_score(
            GradientBoostingRegressor(n_estimators=M, learning_rate=e,
                                      max_depth=3, random_state=0),
            Xb, yb, cv=cv, scoring='neg_mean_squared_error').mean()
        row.append(s)
        if s < best[1]:
            best = ((e, M), s)
    print(f"  {e:>6}  " + "".join(f"{v:>11.4f}" for v in row))
print("  " + "-" * 51)
print(f"  best held-out: eta={best[0][0]}, {best[0][1]} trees -> {best[1]:.4f}")
print()
print(f"  eta=1.0 reaches its best early and then gets WORSE with more trees.")
print(f"  Small eta keeps improving for longer. Shrinkage and number of rounds")
print(f"  are one decision, not two -- which is why tuning them independently")
print(f"  wastes most of a grid ([[hyperparameter-search]]).")

  LEARNING RATE AND NUMBER OF ROUNDS

Training error always prefers the bigger step
     eta      5 trees   20 trees  100 trees
  ----------------------------------------
    0.05      20.4023    10.4408     1.8296
     0.1      15.7170     5.5012     0.8946
     0.5       4.5781     1.0351     0.0302
     1.0       3.9095     0.9189     0.0016
  ----------------------------------------

Held-out error does not
     eta      5 trees   20 trees  100 trees  400 trees
  ---------------------------------------------------
    0.05      21.6237    13.0418     5.8020     4.8544
     0.1      17.5169     8.8375     5.0782     4.8759
     0.5       8.5494     6.8702     6.9667     6.9842
     1.0      10.3849    10.8073    11.2455    11.2466
  ---------------------------------------------------
  best held-out: eta=0.05, 400 trees -> 4.8544

  eta=1.0 reaches its best early and then gets WORSE with more trees.
  Small eta keeps improving for longer. Shrinkage and number of rounds
  are one dec

## 4. Changing the loss changes the algorithm

This is the payoff for phrasing it as *gradients* rather than *residuals*. Swap
the loss and everything else stays put:

| loss | $-\partial L/\partial F$ | behaviour |
|---|---|---|
| squared error $\tfrac12(y-F)^2$ | $y - F$ | the plain residual; sensitive to outliers |
| absolute error $\lvert y-F \rvert$ | $\operatorname{sign}(y - F)$ | every row pulls with equal force — robust |
| Huber | residual, clipped | squared near zero, absolute in the tail |
| log loss (binary) | $y - p$, with $p = \sigma(F)$ | classification, on the log-odds scale |

Look at the absolute-error row. The gradient is $\pm 1$ regardless of *how*
wrong the prediction is, so a wildly outlying point pulls no harder than a
slightly wrong one. That is the entire robustness mechanism, and it falls
straight out of the derivative.

In [5]:
banner("THE SAME MACHINERY, DIFFERENT LOSSES")

step("Pseudo-residuals for one badly outlying point")
y_out = y.copy()
y_out[3] = 60.0                       # was 4.8 -- a data-entry error
F0 = np.full(n_samples, np.median(y_out))
res = y_out - F0
print(f"  recording {rec_ids[3]} corrupted: {y[3]:.1f} -> {y_out[3]:.1f}")
print()
print(f"  {'id':>6}  {'y':>7}  {'y - F':>9}  {'squared-err grad':>17}"
      f"  {'abs-err grad':>13}")
print("  " + "-" * 58)
for i in [0, 2, 3, 5]:
    print(f"  {rec_ids[i]:>6}  {y_out[i]:>7.1f}  {res[i]:>9.4f}"
          f"  {res[i]:>17.4f}  {np.sign(res[i]):>13.1f}")
print("  " + "-" * 58)
print(f"  The outlier's squared-error gradient is {res[3]:.1f} -- it will")
print(f"  dominate the next tree entirely. Its absolute-error gradient is"
      f" {np.sign(res[3]):.0f},")
print(f"  exactly the same pull as any other under-predicted row.")

step("What that does to the fitted model")
print("  12 rows is too few to show this: with 60 trees the model simply")
print("  memorises every row, outlier included, and nothing is damaged.")
print("  So this needs a real dataset and a held-out CLEAN test set.")
print()
Xf, yf = make_friedman1(n_samples=400, noise=1.0, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(Xf, yf, test_size=0.4, random_state=0)
rr = np.random.RandomState(0)
y_corrupt = ytr.copy()
bad = rr.choice(len(ytr), int(0.06*len(ytr)), replace=False)
y_corrupt[bad] += rr.choice([-1, 1], len(bad)) * 60

print(f"  {len(ytr)} training rows, {len(bad)} of them ({len(bad)/len(ytr):.0%})"
      f" corrupted by +/-60")
print(f"  test set is {len(yte)} CLEAN rows the model never sees")
print()
print(f"  {'loss':>16}  {'test MAE':>9}")
print("  " + "-" * 28)
mae = {}
for loss, lab in [('squared_error', 'squared error'),
                  ('huber', 'huber'),
                  ('absolute_error', 'absolute error')]:
    g = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=3,
                                  loss=loss, random_state=0).fit(Xtr, y_corrupt)
    mae[lab] = np.abs(yte - g.predict(Xte)).mean()
    print(f"  {lab:>16}  {mae[lab]:>9.4f}")
g_ref = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=3,
                                  random_state=0).fit(Xtr, ytr)
ref = np.abs(yte - g_ref.predict(Xte)).mean()
print("  " + "-" * 28)
print(f"  {'(no corruption)':>16}  {ref:>9.4f}   <- the target to recover")
print()
print(f"  Squared error is {mae['squared error']/ref:.1f}x the clean baseline:"
      f" {len(bad)} corrupted rows")
print(f"  out of {len(ytr)} dominate every tree, because a residual of 60")
print(f"  contributes a gradient of 60.")
print(f"  Absolute error gets within {mae['absolute error']-ref:.2f} of that")
print(f"  baseline, because those rows pull with force 1, exactly like")
print(f"  everything else. Huber sits between the two, by construction.")
print()
print(f"  ordering as predicted (abs < huber < squared):"
      f" {mae['absolute error'] < mae['huber'] < mae['squared error']}")

step("Binary log loss: the gradient is y - p")
sigmoid = lambda z: 1/(1+np.exp(-z))
p0 = y_class.mean()
F_log = np.full(n_samples, np.log(p0/(1-p0)))          # log-odds of the base rate
p = sigmoid(F_log)
print(f"  base rate = {p0:.4f}  ->  F_0 = log-odds = {F_log[0]:.4f}")
print(f"  p = sigmoid(F_0) = {p[0]:.4f}   (recovers the base rate:"
      f" {np.allclose(p[0], p0)})")
print()
print(f"  {'id':>6}  {'y':>3}  {'p':>8}  {'grad = y - p':>13}")
print("  " + "-" * 36)
for i in range(6):
    print(f"  {rec_ids[i]:>6}  {y_class[i]:>3}  {p[i]:>8.4f}"
          f"  {y_class[i]-p[i]:>13.4f}")
print("  " + "-" * 36)
print(f"  Positives pull up by 1-p, negatives pull down by p. Boosting then")
print(f"  proceeds identically -- fit a tree to that column, add eta * h.")
print(f"  The trees live on the LOG-ODDS scale; sigmoid is applied at the end.")

  THE SAME MACHINERY, DIFFERENT LOSSES

Pseudo-residuals for one badly outlying point
  recording rec03 corrupted: 4.8 -> 60.0

      id        y      y - F   squared-err grad   abs-err grad
  ----------------------------------------------------------
   rec00      0.2    -3.6000            -3.6000           -1.0
   rec02      6.4     2.6000             2.6000            1.0
   rec03     60.0    56.2000            56.2000            1.0
   rec05      9.1     5.3000             5.3000            1.0
  ----------------------------------------------------------
  The outlier's squared-error gradient is 56.2 -- it will
  dominate the next tree entirely. Its absolute-error gradient is 1,
  exactly the same pull as any other under-predicted row.

What that does to the fitted model
  12 rows is too few to show this: with 60 trees the model simply
  memorises every row, outlier included, and nothing is damaged.
  So this needs a real dataset and a held-out CLEAN test set.

  240 training rows,

## 5. Where XGBoost goes further

Everything above uses only the **first** derivative — it is gradient descent in
function space. XGBoost's central change is to use the second derivative too,
making it **Newton's method** in function space.

Expand the loss to second order around the current prediction, with
$g_i = \partial L/\partial F$ and $h_i = \partial^2 L/\partial F^2$:

$$L \approx \sum_i \left[ g_i f(x_i) + \tfrac{1}{2} h_i f(x_i)^2 \right] + \Omega(f)$$

Minimising over a leaf containing rows $I$, with $L_2$ regularisation $\lambda$,
gives the optimal leaf weight directly:

$$w^*_I = -\frac{\sum_{i \in I} g_i}{\sum_{i \in I} h_i + \lambda}$$

Compare with plain gradient boosting, where the leaf value is just the **mean of
the gradients** in that leaf. The denominator is the difference: leaves whose
rows have high curvature get smaller steps. For squared error $h_i = 1$, so
$w^* = -\bar{g}\cdot n/(n+\lambda)$ and the two coincide apart from shrinkage —
which is why the distinction is invisible on regression and matters for
classification, where $h_i = p_i(1-p_i)$ varies a great deal across rows.

In [6]:
banner("FIRST-ORDER vs SECOND-ORDER LEAF VALUES")

step("For squared error the two agree (h = 1 everywhere)")
r_sq = y - np.full(n_samples, y.mean())
g_sq, h_sq = -r_sq, np.ones(n_samples)
grp = np.arange(n_samples) < 6
lam = 0.0
print(f"  leaf with {int(grp.sum())} rows")
print(f"    gradient-boosting leaf = mean(residual)  = {r_sq[grp].mean():+.6f}")
print(f"    XGBoost leaf = -sum(g)/(sum(h)+lambda)   = "
      f"{-g_sq[grp].sum()/(h_sq[grp].sum()+lam):+.6f}")
print(f"    MATCH: {np.allclose(r_sq[grp].mean(), -g_sq[grp].sum()/(h_sq[grp].sum()+lam))}")

step("For log loss they do NOT (h = p(1-p) varies)")
p = sigmoid(np.full(n_samples, np.log(y_class.mean()/(1-y_class.mean()))))
# make the probabilities spread out, as they would be after a few rounds
F_now = np.array([-2.5, -1.8, 2.2, 1.4, -3.0, 2.8, 0.9, 1.9, -2.1, 2.4, -0.6, 1.1])
p = sigmoid(F_now)
g = -(y_class - p)
h = p * (1 - p)
print(f"  {'id':>6}  {'p':>7}  {'g = -(y-p)':>11}  {'h = p(1-p)':>11}")
print("  " + "-" * 40)
for i in range(n_samples):
    print(f"  {rec_ids[i]:>6}  {p[i]:>7.4f}  {g[i]:>11.4f}  {h[i]:>11.4f}")
print("  " + "-" * 40)
print(f"  h ranges from {h.min():.4f} to {h.max():.4f} -- a"
      f" {h.max()/h.min():.0f}x spread.")
print(f"  Confident rows (p near 0 or 1) have tiny curvature and barely move.")
print()
for lam in [0.0, 1.0]:
    first  = -g[grp].mean()
    second = -g[grp].sum()/(h[grp].sum()+lam)
    print(f"  lambda={lam}:  first-order leaf = {first:+.6f}"
          f"   second-order leaf = {second:+.6f}"
          f"   ratio {second/first:.3f}")
print()
print(f"  This is the step from this page to [[xgboost]]: same loop, same")
print(f"  trees, a better leaf value -- plus the regularisation term that")
print(f"  lambda is doing in that denominator.")

  FIRST-ORDER vs SECOND-ORDER LEAF VALUES

For squared error the two agree (h = 1 everywhere)
  leaf with 6 rows
    gradient-boosting leaf = mean(residual)  = -0.208333
    XGBoost leaf = -sum(g)/(sum(h)+lambda)   = -0.208333
    MATCH: True

For log loss they do NOT (h = p(1-p) varies)
      id        p   g = -(y-p)   h = p(1-p)
  ----------------------------------------
   rec00   0.0759       0.0759       0.0701
   rec01   0.1419       0.1419       0.1217
   rec02   0.9002      -0.0998       0.0898
   rec03   0.8022      -0.1978       0.1587
   rec04   0.0474       0.0474       0.0452
   rec05   0.9427      -0.0573       0.0540
   rec06   0.7109      -0.2891       0.2055
   rec07   0.8699      -0.1301       0.1132
   rec08   0.1091       0.1091       0.0972
   rec09   0.9168      -0.0832       0.0763
   rec10   0.3543       0.3543       0.2288
   rec11   0.7503      -0.2497       0.1874
  ----------------------------------------
  h ranges from 0.0452 to 0.2288 -- a 5x spread.
  Co

## 6. What to take away

**"Fit the residual" is a special case, not the definition.** Gradient boosting
fits the negative gradient; for squared error that happens to be the residual.
Phrasing it as a gradient is what lets the same loop do robust regression,
classification, ranking, and survival analysis.

**The loss determines the behaviour.** Squared error chases outliers because its
gradient grows with the error; absolute error ignores them because its gradient
is $\pm 1$. Neither is a special robustness feature — both are just derivatives.

**Shrinkage and rounds are one decision.** A large $\eta$ minimises training
error fastest and generalises worst. Tune the pair together.

**XGBoost is this plus a second derivative.** Leaf value $-\sum g / (\sum h +
\lambda)$ rather than the mean gradient. Identical for squared error, materially
different for log loss.

### Verified in this notebook

- Three hand-built rounds reproduce `GradientBoostingRegressor` to machine precision, including the initial constant
- Each stump's leaf values equal the mean of the residuals falling in that leaf
- $\eta = 1.0$ gets worse with more trees on held-out data while $\eta = 0.05$ keeps improving
- The first- and second-order leaf formulas agree exactly for squared error and diverge for log loss, where curvature spans a large range

### Where this connects

[[adaboost]] is the same idea with exponential loss and row reweighting.
[[xgboost]] adds the second-order step and explicit regularisation;
[[lightgbm]] and [[catboost]] change how the trees are grown and how splits are
found. [[optimisers-sgd-to-adam]] is the same learning-rate argument in
parameter space rather than function space.